Chargement des données pour le traitement

In [1]:
import pandas as pd
try:
    df = pd.read_csv(
        'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
    )
except Exception as e:
    print(f"Une erreur s'est produite lors de la lecture du fichier CSV: {e}")

C:\Users\Djamal TOE\AppData\Local\Temp\ipykernel_21720\912013414.py:3: DtypeWarning: Columns (0: prenom) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [2]:
df.sample(20)

,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix
341636,63,Puy-de-Dôme,41,Biollet,Valérie,PÉCRESSE,4
70712,01,Ain,262,Montluel,Emmanuel,MACRON,831
415913,71,Saône-et-Loire,454,Saint-Martin-du-Mont,Nicolas,DUPONT-AIGNAN,0
106164,02,Aisne,38,Les Autels,Jean,LASSALLE,0
281065,91,Essonne,109,Brières-les-Scellés,Anne,HIDALGO,5
164712,62,Pas-de-Calais,237,Conteville-lès-Boulogne,Marine,LE PEN,86
47654,33,Gironde,113,Cauvignac,Fabien,ROUSSEL,2
388471,02,Aisne,401,Laigny,Nicolas,DUPONT-AIGNAN,3
214737,10,Aube,129,Dosches,Jean-Luc,MÉLENCHON,34
247038,01,Ain,378,Saint-Maurice-de-Gourdans,Anne,HIDALGO,18


In [3]:
df["candidat"] = df["prenom"] + " " + df["nom"]
df.sample(10)

,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix,candidat
44931,27,Eure,11,Amfreville-Saint-Amand,Fabien,ROUSSEL,15,Fabien ROUSSEL
23106,60,Oise,691,Villers-Vermont,Nathalie,ARTHAUD,1,Nathalie ARTHAUD
323789,2A,Corse-du-Sud,17,Appietto,Valérie,PÉCRESSE,34,Valérie PÉCRESSE
425985,09,Ariège,237,Le Puch,NaN,abstentions,8,NaN
319252,07,Ardèche,17,Assions (Les),Valérie,PÉCRESSE,15,Valérie PÉCRESSE
428325,16,Charente,225,Montemboeuf,NaN,abstentions,115,NaN
175175,89,Yonne,461,Villeneuve-l'Archevêque,Marine,LE PEN,219,Marine LE PEN
262887,43,Haute-Loire,231,Salettes,Anne,HIDALGO,1,Anne HIDALGO
348107,79,Deux-Sèvres,242,Voulmentin,Valérie,PÉCRESSE,35,Valérie PÉCRESSE
420326,85,Vendée,76,Cugand,Nicolas,DUPONT-AIGNAN,43,Nicolas DUPONT-AIGNAN


## 2. Comparaison des scores départements aux moyennes nationales.

Q4. Créons un dataframe nommé score_departements stockant, pour chaque département, le nombre de vote obtenu pour chaque candidat et le score (en %).

In [4]:
# Votes par candidat
score_departements = df.groupby(["code_departement", "candidat"]).agg(
    votes = ("voix", "sum")
).reset_index()

# Récupération des totaux sans tenir compte des votes adressé à personne
total_dep = df.loc[~df["candidat"].isna()].groupby(["code_departement"]).agg(
    total = ("voix", "sum")
).reset_index()

# Rajout du total
score_departements = score_departements.merge(right=total_dep, how="left", on="code_departement")
score_departements


# Calcul du score
score_departements["score"] = round(100 * score_departements["votes"] / score_departements["total"], 2)  # .astype(str)  + "%"
score_departements = score_departements.drop(columns="total")

In [5]:
# affichage comme souhaité
x = score_departements.copy()
x["score"] = x["score"].astype(str)  + "%"
x

,code_departement,candidat,votes,score
0,01,Anne HIDALGO,5644,1.69%
1,01,Emmanuel MACRON,92206,27.69%
2,01,Fabien ROUSSEL,5938,1.78%
3,01,Jean LASSALLE,10876,3.27%
4,01,Jean-Luc MÉLENCHON,57832,17.37%
...,...,...,...,...
1291,fr_etranger,Nicolas DUPONT-AIGNAN,7074,1.42%
1292,fr_etranger,Philippe POUTOU,3145,0.63%
1293,fr_etranger,Valérie PÉCRESSE,20956,4.2%
1294,fr_etranger,Yannick JADOT,40774,8.17%


cellule de vérification ci-dessous avec le département 11

In [6]:
x.loc[x["code_departement"] == "11"].sort_values("votes", ascending=False)

,code_departement,candidat,votes,score
125,11,Marine LE PEN,64027,30.14%
121,11,Emmanuel MACRON,43104,20.29%
124,11,Jean-Luc MÉLENCHON,42039,19.79%
131,11,Éric ZEMMOUR,18434,8.68%
123,11,Jean LASSALLE,12382,5.83%
129,11,Valérie PÉCRESSE,7350,3.46%
130,11,Yannick JADOT,6322,2.98%
120,11,Anne HIDALGO,6166,2.9%
122,11,Fabien ROUSSEL,5622,2.65%
127,11,Nicolas DUPONT-AIGNAN,4206,1.98%


Q5. Refaissons le lien avec le niveau national pour comparer le score départemental avec le score national. Et nommons ce dataframe score_departements, nous allons le réutiliser par la suite.

In [7]:
# Votes par candidat
nationale = df.groupby(["candidat"]).agg(
    votes_national = ("voix", "sum")
).reset_index()

# Récupération des totaux sans tenir compte des votes adressé à personne
total_nationale = sum(df.loc[~df["candidat"].isna()]["voix"])

# Calcul du score
nationale["score_national"] = round(100 * nationale["votes_national"] / total_nationale, 2) # .astype(str)  + "%"
nationale

# Jointure avec la table relatives aux départements
score_departements = score_departements.merge(right=nationale, on="candidat", how="left")
score_departements = score_departements.rename(columns={"votes": "votes_departement", "score": "score_departement"})
score_departements

,code_departement,candidat,votes_departement,score_departement,votes_national,score_national
0,01,Anne HIDALGO,5644,1.69,616478,1.75
1,01,Emmanuel MACRON,92206,27.69,9783058,27.85
2,01,Fabien ROUSSEL,5938,1.78,802422,2.28
3,01,Jean LASSALLE,10876,3.27,1101387,3.13
4,01,Jean-Luc MÉLENCHON,57832,17.37,7712520,21.95
...,...,...,...,...,...,...
1291,fr_etranger,Nicolas DUPONT-AIGNAN,7074,1.42,725176,2.06
1292,fr_etranger,Philippe POUTOU,3145,0.63,268904,0.77
1293,fr_etranger,Valérie PÉCRESSE,20956,4.20,1679001,4.78
1294,fr_etranger,Yannick JADOT,40774,8.17,1627853,4.63


In [8]:
# Affichage comme demandé
x = score_departements.copy()
x["score_departement"] = x["score_departement"].astype(str)  + "%"
x["score_national"] = x["score_national"].astype(str)  + "%"
x

,code_departement,candidat,votes_departement,score_departement,votes_national,score_national
0,01,Anne HIDALGO,5644,1.69%,616478,1.75%
1,01,Emmanuel MACRON,92206,27.69%,9783058,27.85%
2,01,Fabien ROUSSEL,5938,1.78%,802422,2.28%
3,01,Jean LASSALLE,10876,3.27%,1101387,3.13%
4,01,Jean-Luc MÉLENCHON,57832,17.37%,7712520,21.95%
...,...,...,...,...,...,...
1291,fr_etranger,Nicolas DUPONT-AIGNAN,7074,1.42%,725176,2.06%
1292,fr_etranger,Philippe POUTOU,3145,0.63%,268904,0.77%
1293,fr_etranger,Valérie PÉCRESSE,20956,4.2%,1679001,4.78%
1294,fr_etranger,Yannick JADOT,40774,8.17%,1627853,4.63%


cellule de vérification ci-dessous avec le département 1

In [9]:
x.loc[x["code_departement"] == "11"].sort_values("votes_departement", ascending=False)

,code_departement,candidat,votes_departement,score_departement,votes_national,score_national
125,11,Marine LE PEN,64027,30.14%,8133828,23.15%
121,11,Emmanuel MACRON,43104,20.29%,9783058,27.85%
124,11,Jean-Luc MÉLENCHON,42039,19.79%,7712520,21.95%
131,11,Éric ZEMMOUR,18434,8.68%,2485226,7.07%
123,11,Jean LASSALLE,12382,5.83%,1101387,3.13%
129,11,Valérie PÉCRESSE,7350,3.46%,1679001,4.78%
130,11,Yannick JADOT,6322,2.98%,1627853,4.63%
120,11,Anne HIDALGO,6166,2.9%,616478,1.75%
122,11,Fabien ROUSSEL,5622,2.65%,802422,2.28%
127,11,Nicolas DUPONT-AIGNAN,4206,1.98%,725176,2.06%


Q6. Créons une variable surrepresentation qui compare, en relatif, les scores nationaux et départementaux

In [10]:
score_departements["surrepresentation"] = round(((score_departements["score_departement"] / score_departements["score_national"]) - 1) * 100, 2)

In [11]:
# Affichage comme demandé (avec le symbole %)
x = score_departements.copy()
x["score_departement"] = x["score_departement"].astype(str)  + "%"
x["score_national"] = x["score_national"].astype(str)  + "%"
x["surrepresentation"] = x["surrepresentation"].astype(str)  + "%"
x

,code_departement,candidat,votes_departement,score_departement,votes_national,score_national,surrepresentation
0,01,Anne HIDALGO,5644,1.69%,616478,1.75%,-3.43%
1,01,Emmanuel MACRON,92206,27.69%,9783058,27.85%,-0.57%
2,01,Fabien ROUSSEL,5938,1.78%,802422,2.28%,-21.93%
3,01,Jean LASSALLE,10876,3.27%,1101387,3.13%,4.47%
4,01,Jean-Luc MÉLENCHON,57832,17.37%,7712520,21.95%,-20.87%
...,...,...,...,...,...,...,...
1291,fr_etranger,Nicolas DUPONT-AIGNAN,7074,1.42%,725176,2.06%,-31.07%
1292,fr_etranger,Philippe POUTOU,3145,0.63%,268904,0.77%,-18.18%
1293,fr_etranger,Valérie PÉCRESSE,20956,4.2%,1679001,4.78%,-12.13%
1294,fr_etranger,Yannick JADOT,40774,8.17%,1627853,4.63%,76.46%


Q7. Créons une fonction pour représenter une figure similaire à Figure 1 pour un candidat donné des
principales surreprésentations (en valeur absolue) par département.

In [14]:
def display_surrepresentation(candidat: str, top: int=5):
    """
        Permet de représenter visuellement la surreprésentation des candidats

        Parameters
        ------------
            candidat : str
                Prenom et nom du candidat
            top : int
                le top a afficher (par défaut 5)

    """

    import matplotlib.pyplot as plt

    # Selection des données pour un candidat donné
    res = score_departements.loc[(
        (score_departements["candidat"] == candidat)
        )].sort_values(by="surrepresentation", key=abs, ascending=False).head(top) # On ordonne selon la valeur absolue de la surreprésentation

    res = res.sort_values("surrepresentation", ascending=True)

    plt.barh(y = res["code_departement"], width=res["surrepresentation"])
    plt.title("Top 5 des surreprésentation de " + candidat)
    plt.axvline(x=0)
    plt.show()

In [15]:
display_surrepresentation(candidat="Éric ZEMMOUR")

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

In [ ]:
display_surrepresentation(candidat="Emmanuel MACRON", top=10)

AttributeError: module 'matplotlib' has no attribute 'get_data_path'

## 3. Un peu de cartographie

### Question 8 : Faire une fonction permettant de restreindre score_departements en fonction d’un candidat. Commencer par tester sur Marine Le Pen (créer un nouvel objet, ne pas écraser score_departements nous allons l’utiliser à nouveau !). Faire une jointure au fond de carte des départements et effectuer une carte de la représentation

In [ ]:
# Importation de la classe Cartographie et téléchargement des données géographiques
from Modules.cartographie import Cartographie
cartographie = Cartographie()
cartographie.download_departement_borders()

AttributeError: module 'matplotlib' has no attribute 'get_data_path'

In [ ]:
# Affichage des résultats pour Marine Le Pen
cartographie.afficher_resultats_filtre_score_candidat(df_score=score_departements, candidat="Marine Le Pen", afficher_carte=True)

In [ ]:
# Affichage des résultats pour Emmanuel Macron
cartographie.afficher_resultats_filtre_score_candidat(df_score=score_departements, candidat="Emmanuel Macron", afficher_carte=True)

In [ ]:
# Affichage des résultats pour Eric Zemmour
cartographie.afficher_resultats_filtre_score_candidat(df_score=score_departements, candidat="Eric Zemmour", afficher_carte=True)